# Lab 3 - Semantic Search (Track N: notebook)

One file, top to bottom. Work the cells in order, run each one, read its
output. Nothing here asks you to open another script.

This repo has two tracks. **This notebook is Track N.** The alternative is
Track S - the script `labs/lab3/search.py`. Pick one and ignore the other
for the whole lab. (You still submit `search.py` + `report.md` either way.)

| Part | What you do | ~time |
|---|---|---|
| 0 | Environment + data check | 10 min |
| Base | Run the provided baseline; write the numbers down | 10 min |
| A | Chunking sweep (four strategies, three sizes) | 40 min |
| B | Dense vs BM25 vs hybrid, **by question kind** | 40 min |
| C | Reranking + the two deployment answers | 35 min |
| D | ANN vs exact; metadata filter | 25 min |
| Submit | Save sweeps + fill `report.md` | 10 min |

**Keep open:** `labs/lab3/CONCEPTS.md`.

**The one rule:** run cells in order, and when a cell says *your turn*, edit it
there.

Every *your turn* cell has **numbered TODOs** (`A1.1`, `A1.2`, ...) with
**commented scaffold lines** to uncomment and adapt. Fill those in; the
`assert` at the bottom of the cell tells you exactly what is still missing.
Answer cells (A4, B2, B5, C3, C4) print a blank template for you to fill and
copy into `report.md`. When you are done, finish `labs/lab3/search.py` and
fill `labs/lab3/report.md` by hand - never regenerate the report.

**Skip-with-note on some Windows ARM64 hosts:** C1 needs
sentence-transformers/torch and D1/D2 need chromadb; those cells detect a
missing package, print a skip message, and you say so in the report. **D3
works either way** (the dense fallback is a YOUR TURN cell).

**The metric trap:** `hit_rate@5` is saturated (0.93-0.98 for every
retriever). Use **nDCG@10, recall@5, hit_rate@1, MRR**.

**n = 42:** Q36, Q38, Q39 have no relevant document; `load_questions()`
already drops them. Say so in your report.


## 0 - Setup

In [ ]:
# Make the shared library importable and load settings. Safe to re-run.
import os, sys, json, statistics, time
from collections import defaultdict
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "aip").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "aip").is_dir():
    ROOT = Path.cwd().parents[1]          # notebook opened from labs/lab3
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")
os.chdir(ROOT)
print("repo root:", ROOT)
print("python:", sys.version.split()[0])

In [ ]:
# Lab 3 dependencies. chromadb (D1/D2) and sentence-transformers (C1) may
# lack a Windows ARM64 wheel - those are skip-with-note, not hard failures.
ok = True
for mod in ("pydantic", "httpx", "dotenv", "numpy", "rank_bm25"):
    try:
        __import__(mod)
        print(f"PASS  import {mod}")
    except ImportError:
        ok = False
        print(f"FAIL  import {mod}  ->  pip install -r requirements.txt")

try:
    import sentence_transformers
    HAS_CE = True
    print("PASS  import sentence_transformers (C1 available)")
except ImportError:
    HAS_CE = False
    print("WARN  sentence-transformers not installed - skip C1, note it in report.md")

try:
    import chromadb
    HAS_CHROMA = True
    print("PASS  import chromadb (D1/D2 available)")
except ImportError:
    HAS_CHROMA = False
    print("WARN  chromadb not installed - skip D1/D2, note it in report.md (D3 still runs)")

key = os.getenv("GEMINI_API_KEY", "")
print(("PASS" if key.strip() else "FAIL"), "GEMINI_API_KEY set")
assert ok and key.strip(), "fix the FAIL lines above before continuing"


In [ ]:
# Library surface for Lab 3. Safe to re-run.
import numpy as np
from aip import cost
from aip.chunking import STRATEGIES, Chunk, markdown_chunks
from aip.evals import retrieval_metrics
from aip.retrieval import (
    Bm25Retriever, ChromaRetriever, DenseRetriever, HybridRetriever,
    CrossEncoderReranker, LLMReranker, Retriever,
)

# Reuse the provided scaffolding from Track S (same file, no duplication).
search = {"__file__": str(ROOT / "labs/lab3/search.py")}
exec((ROOT / "labs/lab3/search.py").read_text(encoding="utf-8").split("if __name__")[0], search)
load_corpus = search["load_corpus"]
load_questions = search["load_questions"]
build_chunks = search["build_chunks"]
evaluate = search["evaluate"]
table = search["table"]
kind_table = search["kind_table"]
record = search["record"]
snippet = search["snippet"]
time_search = search["time_search"]
hit_at_1 = search["hit_at_1"]
strip_heading_prefix = search["strip_heading_prefix"]
index_and_eval = search["index_and_eval"]
best_chunk_cfg = search["best_chunk_cfg"]
print("scaffolding loaded from labs/lab3/search.py")


In [ ]:
# Data check: 30 docs, 45 questions, 42 with a non-empty relevant set.
corpus = load_corpus()
questions_all = load_questions(include_unanswerable=True)
questions = load_questions()
print(f"corpus: {len(corpus)} docs")
print(f"golden: {len(questions_all)} rows -> {len(questions)} used for retrieval metrics")
kinds = sorted({q["kind"] for q in questions_all})
print("kinds:", kinds)
archived = [d for d in corpus if "ARCHIVED" in d]
print("ARCHIVED docs (Part D trap):", archived)
assert len(corpus) == 30 and len(questions_all) == 45 and len(questions) == 42

## Baseline - write these numbers down

Sliding-800 + dense exact search. First run embeds 30 docs (1-3 min);
after that the cache makes it instant.

In [ ]:
chunks0 = build_chunks(corpus, "sliding", 800, overlap=150)
print(f"{len(corpus)} docs -> {len(chunks0)} chunks")
dense0 = DenseRetriever(chunks0)   # embeds (cached after first time)
m0 = evaluate(dense0, questions)
print(table({"baseline sliding-800 dense": m0}))
print()
print(kind_table(m0))
print("\nWrite nDCG@10, recall@5, hit_rate@1, mrr into report.md section 0 NOW.")
record("baseline", {"config": "sliding-800 dense", "n_chunks": len(chunks0), "metrics": m0})

## Part A - Chunking (40 min)

**A1** four strategies at 800. **A2** winner at 400/800/1600.
**A3** markdown prefix on/off. **A4** one failure case.

Report chunk count and build time with quality.

**A1-A3 are YOUR TURN cells** with numbered steps (`A1.1`-`A1.5`, ...).
Uncomment the scaffold lines, fill them in, and re-run - the `assert` tells
you what is still missing. **A4** finds the failing question for you - you
write the diagnosis from the printed template.


In [ ]:
cols = ("hit_rate@1", "hit_rate@5", "recall@5", "mrr", "ndcg@10")

def run_cfg(strategy, size, **kw):
    """Provided: chunk -> embed (cached after first time) -> evaluate.
    Returns (chunks, retriever, metrics)."""
    t0 = time.perf_counter()
    ch = build_chunks(corpus, strategy, size, **kw)
    build_s = time.perf_counter() - t0
    t1 = time.perf_counter()
    r = DenseRetriever(ch, show_progress=False)
    index_s = time.perf_counter() - t1
    m = evaluate(r, questions)
    print(f"  {strategy}-{size}: chunks={len(ch)} build={build_s:.1f}s index={index_s:.1f}s")
    return ch, r, m

# YOUR TURN - A1: four strategies at size=800 (README A1 / search.py TODO A1).
#
#   A1.1  fixed-800       run_cfg("fixed", 800)
#   A1.2  sliding-800     run_cfg("sliding", 800, overlap=150)
#   A1.3  recursive-800   run_cfg("recursive", 800, overlap=100)
#   A1.4  markdown-800    run_cfg("markdown", 800)
#   A1.5  print the table
#
# Each call returns (chunks, retriever, metrics). For every strategy store:
#     rows_a1["name"] = metrics
#     cfgs["name"]    = (chunks, retriever)   # Part A4 reuses these
#
# Checkpoint: markdown-800 must beat fixed-800 by a clear margin on nDCG@10.
# If all four land within ~1 point, your chunker is not being applied - a
# bug, not a finding.
#
# Report nDCG@10, recall@5, hit_rate@1, MRR, chunk count, index seconds
# into report.md A1.

print("== A1 four strategies @800 ==")
rows_a1 = {}
cfgs = {}

# A1.1 - uncomment, then repeat the pattern for A1.2-A1.4:
# ch, r, m = run_cfg("fixed", 800)
# rows_a1["fixed-800"] = m
# cfgs["fixed-800"] = (ch, r)

# A1.2
# ch, r, m = run_cfg("sliding", 800, overlap=150)
# rows_a1["sliding-800"] = m
# cfgs["sliding-800"] = (ch, r)

# A1.3
# ch, r, m = run_cfg("recursive", 800, overlap=100)
# rows_a1["recursive-800"] = m
# cfgs["recursive-800"] = (ch, r)

# A1.4
# ch, r, m = run_cfg("markdown", 800)
# rows_a1["markdown-800"] = m
# cfgs["markdown-800"] = (ch, r)

# A1.5
# print(table(rows_a1, cols))

assert len(rows_a1) == 4 and len(cfgs) == 4, (
    "TODO A1 incomplete: uncomment the four run_cfg blocks above "
    "(A1.1-A1.4) so rows_a1 and cfgs each have 4 entries, then re-run."
)
print(table(rows_a1, cols))


In [ ]:
# YOUR TURN - A2: size sweep on the A1 winner (README A2 / search.py TODO A2).
#
#   A2.1  Pick the winner: highest ndcg@10 in rows_a1 (done below as w_strat)
#   A2.2  Run that strategy at size 400, 800, 1600 via run_cfg()
#   A2.3  Store rows_a2[f"{w_strat}-{size}"] = metrics
#   A2.4  print the table
#
# The curve is NOT monotonic - explain the shape (dilution vs incomplete
# rules, CONCEPTS.md) in report.md A2.

assert len(globals().get("rows_a1") or {}) == 4, (
    "Complete the A1 YOUR TURN cell first (rows_a1 needs 4 entries)."
)
winner = max(rows_a1, key=lambda k: rows_a1[k]["ndcg@10"])
w_strat = winner.rsplit("-", 1)[0]
kw = {"sliding": {"overlap": 150}, "recursive": {"overlap": 100}}.get(w_strat, {})
print(f"== A2 size sweep on A1 winner: {winner} ==")
rows_a2 = {}

# A2.2 size 400
# ch, r, m = run_cfg(w_strat, 400, **kw)
# rows_a2[f"{w_strat}-400"] = m

# A2.2 size 800
# ch, r, m = run_cfg(w_strat, 800, **kw)
# rows_a2[f"{w_strat}-800"] = m

# A2.2 size 1600
# ch, r, m = run_cfg(w_strat, 1600, **kw)
# rows_a2[f"{w_strat}-1600"] = m

# A2.4
# print(table(rows_a2, cols))

assert len(rows_a2) == 3, (
    "TODO A2 incomplete: sweep the A1 winner at 400/800/1600 so rows_a2 "
    "has 3 entries, then re-run."
)
print(table(rows_a2, cols))


In [ ]:
# YOUR TURN - A3: markdown WITH vs WITHOUT the '[heading > path]' prefix.
# README A3 / search.py TODO A3. Do NOT edit aip/chunking.py - use the
# provided strip_heading_prefix() instead.
#
#   A3.1  Build markdown chunks at best_size, evaluate -> m_on
#   A3.2  Strip the prefix, re-index, evaluate -> m_off
#   A3.3  print both rows + delta on nDCG@10, hit_rate@1, hit_rate@5
#
# hit_rate@5 may FALL while hit_rate@1 RISES - work out which improved.
# Reference delta is in search.py TODO A3.

assert len(globals().get("rows_a2") or {}) == 3, (
    "Complete the A2 YOUR TURN cell first (rows_a2 needs 3 entries)."
)
best_size = int(max(rows_a2, key=lambda k: rows_a2[k]["ndcg@10"]).rsplit("-", 1)[1])
print(f"== A3 markdown@{best_size} prefix on/off ==")
m_on, m_off = None, None

# A3.1 prefix ON
# ch_on, r_on, m_on = run_cfg("markdown", best_size)

# A3.2 prefix OFF - strip_heading_prefix(chunks) -> list[Chunk], then re-index
# ch_off = strip_heading_prefix(ch_on)
# r_off, m_off = index_and_eval(ch_off, questions)

# A3.3
# print(table({"prefix": m_on, "no-prefix": m_off}, cols))
# for key in ("ndcg@10", "hit_rate@1", "hit_rate@5"):
#     print(f"  delta {key}: {m_on[key] - m_off[key]:+.4f}")

assert m_on is not None and m_off is not None, (
    "TODO A3 incomplete: evaluate markdown with and without the heading "
    "prefix so m_on and m_off are both set (uncomment A3.1 and A3.2)."
)
print(table({"prefix": m_on, "no-prefix": m_off}, cols))
for key in ("ndcg@10", "hit_rate@1", "hit_rate@5"):
    print(f"  delta {key}: {m_on[key] - m_off[key]:+.4f}")


In [ ]:
# A4 provided: the question whose per-question MRR swings most across the
# four @800 strategies, plus what ranked vs what should have matched.
# YOUR TURN: turn this into report.md A4 (failure case + diagnosis).
assert len(globals().get("rows_a1") or {}) == 4, "Complete the A1 YOUR TURN cell first (need all four @800 rows)."
per800 = {name: m["_per_question_mrr"] for name, m in rows_a1.items()}

def spread(qid):
    vals = [per800[n].get(qid, 0.0) for n in rows_a1]
    return max(vals) - min(vals)

qid = max((q["id"] for q in questions), key=spread)
q = next(x for x in questions if x["id"] == qid)
worst = min(rows_a1, key=lambda n: per800[n].get(qid, 0.0))
bestn = max(rows_a1, key=lambda n: per800[n].get(qid, 0.0))
print(f"{qid} ({q['kind']}): {q['question']}")
print("relevant:", q["relevant_docs"])
print("gold:", snippet(q.get("gold_answer", ""), 140))
print("MRR by @800 strategy:", {n: round(per800[n].get(qid, 0), 2) for n in rows_a1})
print(f"worst={worst}  best={bestn}")

ch_w, r_w = cfgs[worst]
print(f"-- {worst} top-3 --")
for h in r_w.search(q["question"], k=3):
    print(f"  rank{h.rank+1} {h.doc_id}: {snippet(h.text)}")
sub = [c for c in ch_w if c.doc_id in set(q["relevant_docs"])]
if sub:
    g = DenseRetriever(sub, show_progress=False).search(q["question"], k=1)[0]
    print(f"  should have matched ({g.doc_id}): {snippet(g.text)}")

# YOUR TURN - A4: fill each line from the output above, copy into report.md.
A4_ANSWER = """
Question id / kind:
Relevant docs / gold snippet:
Chunk that should have matched:
Chunks that did (and why they are wrong):
Diagnosis:
""".strip()
print()
print(A4_ANSWER)

assert globals().get("m_on") is not None, "Complete the A3 YOUR TURN cell first."
record("chunking", {"a1": rows_a1, "a2": rows_a2,
                     "a3": {"on": m_on, "off": m_off},
                     "a4": {"qid": qid, "worst": worst, "best": bestn}})


In [ ]:
# Quick check - Part A must be finished before Part B means anything.
# (YOUR TURN cells assert what is still missing; this cell only checks
# that the results actually landed.)
try:
    assert len(rows_a1) == 4 and len(rows_a2) >= 1
    assert globals().get("m_on") is not None
    print("Part A complete:", sorted(rows_a1), "->", sorted(rows_a2))
except (NameError, AssertionError):
    print("Complete the Part A YOUR TURN cells (A1-A3) first.")


## Part B - Dense vs BM25 vs hybrid (40 min)

**Use MRR / nDCG@10 for the by-kind tables.** `hit_rate@5` is saturated.

Watch **Q44** (exact identifier) and **Q41** (paraphrase, no lexical overlap).

**B2 and B5 are YOUR TURN** answer cells - blank templates with numbered
fields for you to fill and copy into `report.md`. B1, B3 and B4 are
provided runs.


In [ ]:
# B1 provided: freeze YOUR Part A winner (greedy sweep - one axis at a time)
# and run dense / bm25 / hybrid on it. Falls back to markdown-400 (the
# search.py best_chunk_cfg default) if Part A is unfinished.
rows_a2 = globals().get("rows_a2") or {}
if rows_a2:
    win_name = max(rows_a2, key=lambda k: rows_a2[k]["ndcg@10"])
else:
    win_name = "markdown-400"
    print("Part A not finished - defaulting to markdown-400 (search.py default).")
print("using chunking:", win_name)
strat, size_s = win_name.rsplit("-", 1)
kw = {"sliding": {"overlap": 150}, "recursive": {"overlap": 100}}.get(strat, {})
chunks = build_chunks(corpus, strat, int(size_s), **kw)
dense = DenseRetriever(chunks, show_progress=False)
bm25 = Bm25Retriever(chunks)

print("== B1 dense / bm25 / hybrid ==")
m_d = evaluate(dense, questions)
m_b = evaluate(bm25, questions)
m_h = evaluate(HybridRetriever([dense, bm25]), questions)
print(table({"dense": m_d, "bm25": m_b, "hybrid": m_h}, cols))


In [ ]:
# B2 harness - kind breakdown with MRR + Q44/Q41 numbers (the important part).
for name, m in [("dense", m_d), ("bm25", m_b), ("hybrid", m_h)]:
    print(f"-- {name} by kind (mrr) --")
    print(kind_table(m, col="mrr"))
    pq = m["_per_question_mrr"]
    print(f"  {name}: Q44={pq.get('Q44', 0):.2f}  Q41={pq.get('Q41', 0):.2f}")

# YOUR TURN - B2: fill each ___ from the harness above, copy into report.md.
# Use MRR (never hit_rate@5). Two sentences of mechanism per question.
B2_ANSWER = """
Q44 (identifier AUR-HI-SIL-2026):
  MRR dense=___  bm25=___  hybrid=___
  mechanism (2 sentences): ...

Q41 (paraphrase, no lexical overlap):
  MRR dense=___  bm25=___  hybrid=___
  mechanism (2 sentences): ...

hybrid trade: rescued ___ / damaged ___
""".strip()
print()
print(B2_ANSWER)


In [ ]:
# B3 - RRF k sensitivity
print("== B3 rrf_k ==")
rrf_rows = {}
for k in (10, 30, 60, 100):
    rrf_rows[f"hybrid-k{k}"] = evaluate(HybridRetriever([dense, bm25], rrf_k=k), questions)
print(table(rrf_rows, cols))
print("Expect nearly flat - insensitivity is the point.")

# B4 - unequal weights
print("== B4 weights ==")
w_rows = {
    "1:1": m_h,
    "densex2": evaluate(HybridRetriever([dense, bm25], weights=[2.0, 1.0]), questions),
    "bm25x2": evaluate(HybridRetriever([dense, bm25], weights=[1.0, 2.0]), questions),
}
print(table(w_rows, cols))

In [ ]:
# B5 harness - dense vs BM25 vs hybrid, counted on per-question MRR.
pq_d, pq_b, pq_h = m_d["_per_question_mrr"], m_b["_per_question_mrr"], m_h["_per_question_mrr"]
print(f"ndcg@10: dense={m_d['ndcg@10']:.3f}  bm25={m_b['ndcg@10']:.3f}  hybrid={m_h['ndcg@10']:.3f}")
differ = [qid for qid in pq_d if pq_d[qid] != pq_b.get(qid, 0.0)]
d_ahead = sum(1 for qid in differ if pq_d[qid] > pq_b.get(qid, 0.0))
print(f"dense vs bm25 differ on {len(differ)} questions; dense ahead on {d_ahead}")
d_over_h = sum(1 for qid in pq_d if pq_d[qid] > pq_h.get(qid, 0.0))
h_over_d = sum(1 for qid in pq_h if pq_h[qid] > pq_d.get(qid, 0.0))
print(f"vs hybrid: dense better {d_over_h}, hybrid better {h_over_d}")

# YOUR TURN - B5: fill each ___ from the harness above, copy into report.md.
# Report WHICHEVER WAY IT LANDS. Do not suppress a loss; do not dress up a
# tie as a win.
B5_ANSWER = """
nDCG@10: dense=___  hybrid=___  -> hybrid is better / worse / tied because:
Questions where dense beat BM25 (among those they differ): ___/___
""".strip()
print()
print(B5_ANSWER)
record("retrieval", {"chunking": win_name,
                      "b1": {"dense": m_d, "bm25": m_b, "hybrid": m_h},
                      "b3": rrf_rows, "b4": w_rows,
                      "b5": {"differ": len(differ), "dense_ahead": d_ahead}})


## Part C - Reranking (35 min)

Retrieve k=30, rerank to 5. Report quality **and** latency; for the LLM
reranker also **cost** (`cost.global_budget()`).

C3 needs **two different** deployment answers (interactive vs batch).

In [ ]:
cols_c = ("ndcg@5", "hit_rate@1", "recall@5", "mrr", "ndcg@10", "latency_p95_ms")

# Baseline for deltas: retrieve wide, keep natural order.
print("== no rerank (retrieve k=30) ==")
m_none = evaluate(dense, questions, k=30)
print(table({"no-rerank k30": m_none}, cols_c))

# C1 provided runner: cross-encoder (~90 MB download on first run).
# Skip-with-note if sentence-transformers/torch will not install (cell 0).
m_ce = None
if not HAS_CE:
    print("C1 skipped: sentence-transformers not installed - note it in report.md, lean on C2.")
else:
    print("== C1 cross-encoder (k=30 -> 5) ==")
    try:
        t0 = time.perf_counter()
        ce = CrossEncoderReranker()
        print(f"  ready in {time.perf_counter() - t0:.1f}s")
        m_ce = evaluate(dense, questions, k=30, reranker=ce, final_k=5)
        print(table({"no-rerank": m_none, "cross-encoder": m_ce}, cols_c))
        for key in ("ndcg@5", "hit_rate@1", "recall@5", "latency_p95_ms"):
            print(f"  {key:<16} {m_ce[key] - m_none[key]:+.4f}")
    except Exception as exc:
        m_ce = None
        print(f"C1 failed ({type(exc).__name__}: {exc}) - skip C1, note it in report.md.")


In [ ]:
# C2 - LLM reranker: quality + latency + COST (30 sequential calls/question).
# The slow path: 42 questions x 30 calls IN SERIAL - run it once, late.
# If the free tier 429s, wait and retry, or use tier MAIN and report the
# higher $/1k (search.py TODO C2).
m_ce = globals().get("m_ce")
llm_rr = LLMReranker(tier="SMALL")
b0 = cost.global_budget().as_dict()
m_llm = evaluate(dense, questions, k=30, reranker=llm_rr, final_k=5)
b1 = cost.global_budget().as_dict()
n = len(questions)
spend = b1["cost_usd"] - b0["cost_usd"]
llm_calls = b1["calls"] - b0["calls"]
usd_per_1k = spend / n * 1000
rows_c2 = {"no-rerank": m_none}
if m_ce is not None:
    rows_c2["cross-encoder"] = m_ce
rows_c2["llm"] = m_llm
print(table(rows_c2, cols_c))
print(f"  cost: {llm_calls} calls, ${spend:.4f} total -> ${usd_per_1k:.3f}/1k queries")


In [ ]:
# C3 harness: the decision table (missing configs print as skipped).
m_ce = globals().get("m_ce")
m_llm = globals().get("m_llm")
usd_per_1k = globals().get("usd_per_1k")
cost_cell = {
    "no-rerank": "$0.000",
    "cross-encoder": "$0.000 (local)" if m_ce is not None else "skipped",
    "llm": f"${usd_per_1k:.3f}" if usd_per_1k is not None else "not run",
}
print("== C3 decision table ==")
for name, m in [("no-rerank", m_none), ("cross-encoder", m_ce), ("llm", m_llm)]:
    if m is None:
        print(f"  {name:<16} (not run)")
        continue
    print(f"  {name:<16} ndcg@5={m['ndcg@5']:.4f}  hit@1={m['hit_rate@1']:.4f}"
          f"  p95={m['latency_p95_ms']:.0f}ms  {cost_cell.get(name, '')}")

# YOUR TURN - C3: TWO different answers, both defended (README C3).
# (a) interactive agent-facing search box - latency budget first.
# (b) overnight batch job - quality/cost budget first.
# If they come out the same, explain why the budgets do not change the call.
# Fill each ___ from the table above, copy into report.md.
C3_INTERACTIVE = """
(a) Interactive agent-facing search box -> config: ___
because (latency budget first): ...
""".strip()
C3_BATCH = """
(b) Overnight batch job -> config: ___
because (quality/cost budget first): ...
""".strip()
print()
print(C3_INTERACTIVE)
print()
print(C3_BATCH)


In [ ]:
# C4 harness: queries where reranking made per-question MRR WORSE vs no-rerank.
m_ce = globals().get("m_ce")
m_llm = globals().get("m_llm")
spend = globals().get("spend")
usd_per_1k = globals().get("usd_per_1k")
pq_none = m_none["_per_question_mrr"]
drops = []
for label, m in [("cross-encoder", m_ce), ("llm", m_llm)]:
    if m is None:
        continue
    for qid, v in m["_per_question_mrr"].items():
        d = pq_none.get(qid, 0.0) - v
        if d > 0:
            drops.append((d, label, qid))
worst_drop, worst_label, worst_q = 0.0, "", None
if not drops:
    print("no question got worse - report that too (it is a finding)")
else:
    drops.sort(reverse=True)
    worst_drop, worst_label, worst_q = drops[0]
    q = next(x for x in questions if x["id"] == worst_q)
    print(f"worst: {worst_q} under {worst_label}  drop={worst_drop:.2f}")
    print("Q:", q["question"])
    print("relevant:", q["relevant_docs"])

# YOUR TURN - C4: fill each line from the harness above, copy into report.md.
# Was the relevant doc in the k=30 pool? Check the no-rerank ranking for
# this qid - if it never entered the pool, reranking could not have saved it.
C4_ANSWER = """
Question id:
MRR before -> after:
Was the relevant doc in the k=30 pool? Y/N
Diagnosis (failure mode): ...
""".strip()
print()
print(C4_ANSWER)
record("rerank", {"none": m_none, "ce": m_ce, "llm": m_llm,
                   "llm_cost_usd": spend, "usd_per_1k": usd_per_1k,
                   "c4_worst": worst_q})


## Part D - Index and metadata (25 min)

**D1/D2 need `chromadb`.** If the import failed in cell 0, those cells print
a skip message - do Parts A-C, skip D1/D2, and note it in your report.

**D3 works either way.** The dense fallback (a `WhereDense` wrapper) is a
YOUR TURN cell with two numbered paths (A = chroma, B = dense). The lesson is
about labelling at ingest and filtering at query time, not about HNSW.


In [ ]:
# Shared prep for D1-D3 (always runs - D3 needs status metadata either way).
# Uses the chunking you froze in Part B.
chunks_d = build_chunks(corpus, strat, int(size_s), **kw)
for c in chunks_d:
    c.meta["status"] = "archived" if "ARCHIVED" in c.doc_id else "current"
dense_d = DenseRetriever(chunks_d, show_progress=False)

d1_payload = None
d2_payload = None
chroma = None

if not HAS_CHROMA:
    print("chromadb not available - skip D1/D2, note it in report.md (D3 below still runs).")
else:
    chroma = ChromaRetriever(chunks_d, path=".chroma", collection="lab3-best", reset=True)
    m_exact = evaluate(dense_d, questions)
    m_chroma = evaluate(chroma, questions)
    print("== D1 exact vs HNSW ==")
    print(table({"exact": m_exact, "hnsw": m_chroma}, cols))
    for key in ("ndcg@10", "recall@5", "hit_rate@1", "latency_p95_ms"):
        print(f"  gap {key:<16} {m_chroma[key] - m_exact[key]:+.4f}")
    d1_payload = {"exact": m_exact, "chroma": m_chroma}


In [ ]:
# D2 provided: scale with random unit-vector ballast (index only - no golden
# answers, excluded from quality reporting). Time exact vs HNSW at
# ~160 / ~4k / ~40k. time_search() is provided.
# YOUR TURN after it runs: find the crossover and explain the mechanism in
# report.md D2 - at small n HNSW is SLOWER (graph traversal + Python overhead
# vs one BLAS matmul).
if not HAS_CHROMA or chroma is None:
    print("skip D2 (needs chromadb) - note it in report.md")
else:
    n_real = len(chunks_d)
    targets = [n_real, 4000, 40000]
    rng = np.random.default_rng(0)
    max_extra = max(targets) - n_real
    dim = dense_d.matrix.shape[1]
    ball = rng.standard_normal((max_extra, dim), dtype=np.float32)
    ball /= np.linalg.norm(ball, axis=1, keepdims=True)
    ballast = [Chunk(f"Ballast filler {i}", f"ballast-{i}", f"ballast::{i}",
                     {"strategy": "ballast", "status": "ballast"})
               for i in range(max_extra)]

    real_chunks, real_mat = list(dense_d.chunks), dense_d.matrix.copy()
    scale = ChromaRetriever(chunks_d, path=".chroma", collection="lab3-scale", reset=True)
    timing = []
    injected = 0
    for target in targets:
        extra = max(0, target - n_real)
        dense_d.matrix = np.vstack([real_mat, ball[:extra]]) if extra else real_mat
        dense_d.chunks = real_chunks + ballast[:extra]
        if extra > injected:
            t0 = time.perf_counter()
            for i in range(injected, extra, 2000):
                part = ballast[i:min(i + 2000, extra)]
                scale.col.add(
                    ids=[c.chunk_id for c in part],
                    documents=[c.text for c in part],
                    embeddings=ball[i:min(i + 2000, extra)].tolist(),
                    metadatas=[{"doc_id": c.doc_id, "status": "ballast"} for c in part],
                )
            injected = extra
            print(f"  ballast -> {n_real + injected} chunks in {time.perf_counter()-t0:.1f}s")
        e50, e95 = time_search(dense_d, questions)
        h50, h95 = time_search(scale, questions)
        timing.append({"n": n_real + extra, "exact_p95": e95, "hnsw_p95": h95})
        print(f"  n={n_real+extra:>6}  exact p95={e95:7.2f}ms  hnsw p95={h95:7.2f}ms  ratio={h95/e95:.2f}x")
    dense_d.matrix, dense_d.chunks = real_mat, real_chunks
    cross = next((t["n"] for t in timing if t["hnsw_p95"] < t["exact_p95"]), None)
    print("crossover:", cross or "not reached within 40k")
    d2_payload = {"timing": timing, "crossover": cross}


In [ ]:
# D3 - THE TRAP. Q29/Q30/Q31: correct answer in claims-timelines, wrong one
# in claims-timelines-2024-ARCHIVED. Status metadata is stamped in the D1 cell.
# Runs on chroma OR exact dense - no chromadb needed for the lesson.
#
#   D3.1  BEFORE: hit_at_1(retriever, trap_qs)  [already run below]
#   D3.2  AFTER:  hit_at_1(..., where={"status": "current"})
#           Path A (HAS_CHROMA): pass where= to ChromaRetriever directly
#           Path B (no chromadb): wrap DenseRetriever in WhereDense (scaffold)
#   D3.3  print before/after + fill D3_ANSWER (runs once paths set)
#
# This fix required NO change to the retriever. What does that tell you
# about where to look first when retrieval is poor?

trap_qs = [q for q in questions if q["id"] in ("Q29", "Q30", "Q31")]
retriever_d = chroma if (HAS_CHROMA and chroma is not None) else dense_d

# D3.1 provided:
before, per_b = hit_at_1(retriever_d, trap_qs)
print("== D3 status filter (Q29/Q30/Q31) ==")
print(f"  before: hit_rate@1={before:.3f}  {per_b}")

# D3.2 YOUR TURN - uncomment ONE path:
#
# Path A (HAS_CHROMA):
# after, per_a = hit_at_1(retriever_d, trap_qs, where={"status": "current"})
#
# Path B (no chromadb) - DenseRetriever.search has no where= (aip/retrieval.py),
# so wrap it. hit_at_1 calls search(..., k=5, where=...). Over-fetch, filter
# on chunk.meta, cut back to k:
# class WhereDense:
#     def __init__(self, inner):
#         self.inner = inner
#     def search(self, q, k=10, where=None):
#         hits = self.inner.search(q, k=max(k * 5, 30))
#         if where:
#             hits = [h for h in hits
#                     if all(h.chunk.meta.get(kk) == vv
#                            for kk, vv in where.items())]
#         return hits[:k]
# after, per_a = hit_at_1(WhereDense(dense_d), trap_qs,
#                         where={"status": "current"})

after = globals().get("after")
per_a = globals().get("per_a")
if after is None:
    raise NotImplementedError(
        "TODO D3: uncomment Path A (chroma) or Path B (WhereDense wrapper) "
        "above so after/per_a are set, then re-run."
    )

# D3.3 - runs automatically once after/per_a are set:
print(f"  after : hit_rate@1={after:.3f}  {per_a}")
D3_ANSWER = """
What this tells you about where to look first when retrieval is poor:
""".strip()
print(D3_ANSWER)
record("index", {"d1": d1_payload, "d2": d2_payload,
                  "d3": {"before": before, "after": after,
                         "per_b": per_b, "per_a": per_a}})


## Submit

1. Finish the four sweep functions in `labs/lab3/search.py`
   (mirror what you did in the notebook; `record(...)` writes
   `reports/lab3_sweeps.json`).
2. Fill `labs/lab3/report.md` **by hand** - never regenerate it.
3. State n = 42 and why (Q36/Q38/Q39 excluded).
4. Report hybrid **whichever way it lands**.
5. Two different C3 deployment answers.

Run the check cell below before you export the report.

**Also keep open while writing:** `RUNSHEET.md` (the actions, step by step)
and `RUNSHEET.md` (step-by-step actions).


In [ ]:
# Submit check - all five sweeps recorded, no leftover search.py TODOs.
try:
    data = json.loads((ROOT / "reports/lab3_sweeps.json").read_text(encoding="utf-8"))
    missing = [k for k in ("baseline", "chunking", "retrieval", "rerank", "index")
               if k not in data]
    print("sweeps missing from reports/lab3_sweeps.json:", missing or "none - all recorded")
except FileNotFoundError:
    print("reports/lab3_sweeps.json not found - run the baseline cell first.")

src = (ROOT / "labs/lab3/search.py").read_text(encoding="utf-8")
left = src.count("raise NotImplementedError")
print(f"search.py: {left} NotImplementedError stub(s) left "
      f"({'DONE - ready to submit' if left == 0 else 'finish the TODOs before submitting'})")
